# Solution B1 · Preprocessing a mixed dataset, one column type at a time

**Type**: Implement with scikit-learn

## Objective

Turn a dataset with numeric, nominal and ordinal columns into a numeric matrix, using the right scaler and the two
encoders for each group, keeping the column names, and train a logistic regression.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

df = ds.loans(with_missing=False)
NUM = ["age", "income", "job_tenure", "amount"]
NOMINAL = ["city", "contract_type"]
ORDINAL = ["education"]
X = df[NUM + NOMINAL + ORDINAL]
y = df["defaulted"]

## Commented solution

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

scaler = StandardScaler().fit(X_tr[NUM])
Z_num_tr, Z_num_te = scaler.transform(X_tr[NUM]), scaler.transform(X_te[NUM])

ohe = OneHotEncoder(sparse=False, handle_unknown="ignore").fit(X_tr[NOMINAL])
Z_nom_tr, Z_nom_te = ohe.transform(X_tr[NOMINAL]), ohe.transform(X_te[NOMINAL])

oe = OrdinalEncoder(categories=[ds.EDUCATION_ORDER]).fit(X_tr[ORDINAL])
Z_ord_tr, Z_ord_te = oe.transform(X_tr[ORDINAL]), oe.transform(X_te[ORDINAL])

In [ ]:
Z_tr = np.hstack([Z_num_tr, Z_nom_tr, Z_ord_tr])
Z_te = np.hstack([Z_num_te, Z_nom_te, Z_ord_te])
column_names = NUM + list(ohe.get_feature_names_out(NOMINAL)) + ORDINAL
print("shape:", Z_tr.shape, "| first columns:", column_names[:6], "...")

In [ ]:
model = LogisticRegression(max_iter=1000).fit(Z_tr, y_tr)
test_auc = roc_auc_score(y_te, model.predict_proba(Z_te)[:, 1])
coef = pd.Series(model.coef_[0], index=column_names)
top3 = coef.abs().nlargest(3).index.tolist()
print(f"test AUC: {test_auc:.3f}")
print(coef[top3].round(2).to_string())

## Check

In [ ]:
with verify():
    check(len(X_te) == 375 and abs(y_te.mean() - y.mean()) < 0.01, "stratified 75/25 split", "use test_size=0.25 and stratify=y")
    check(Z_tr.shape[1] == 4 + 12 + 4 + 1 and Z_te.shape[1] == Z_tr.shape[1], "21 columns in both training and test",
          "4 numeric + 12 cities + 4 contracts + 1 ordinal = 21")
    check(np.abs(Z_tr[:, :4].mean(axis=0)).max() < 1e-9 and np.abs(Z_te[:, :4].mean(axis=0)).max() > 1e-3,
          "the numeric columns are centered on the training set, not on the test set", "fit the scaler only on X_tr, then transform X_te")
    check(sorted(np.unique(Z_tr[:, -1])) == [0, 1, 2, 3, 4], "the ordinal uses the codes 0-4 in the right order",
          "use OrdinalEncoder(categories=[ds.EDUCATION_ORDER])")
    check(set(np.unique(Z_tr[:, 4:20])) <= {0.0, 1.0}, "the one-hot columns contain only 0 and 1", "columns 4-19 must be one-hot")
    check(len(column_names) == 21 and column_names[:4] == NUM and column_names[-1] == "education" and "contract_type_unemployed" in column_names,
          "column_names is correct", "numeric, then one-hot (get_feature_names_out), then 'education'")
    check(test_auc > 0.65, "the model has a reasonable test AUC", "test_auc should exceed 0.65")
    check(len(top3) == 3 and set(top3) <= set(column_names), "top3 is a list of three column names", "top3 = names of the three columns with the largest |coef|")
done()

## What you learned

- For every group of columns the cycle is the same: `fit` on the training set, `transform` on training and test. If you get the test
  set wrong (`fit_transform`) you have a bug that exercises E1-E3 make you hunt for.
- The transformed matrices lose the names: they must be rebuilt in the order in which you concatenated the blocks. An order
  mistake produces coefficients attributed to the wrong variable (exercise E2).
- A linear model gets a decent AUC with a few lines of preprocessing: the cost of doing it by hand is the effort
  of keeping three `fit` calls aligned, and it is the reason why module 3 introduces `ColumnTransformer` and `Pipeline`.

## Common pitfall

Concatenating the blocks in a different order between training and test (for example `np.hstack([Z_nom_te, Z_num_te, ...])`). The
model does not complain because the number of columns matches, but it applies every coefficient to the wrong column.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why do you encode `education` with an OrdinalEncoder and not with a one-hot?</summary>

Because it has a natural order (middle school < ... < phd) and a single column with those codes lets a linear model
exploit the progression with a single coefficient, with fewer parameters and less noise. A one-hot ignores the
order and adds columns. If I suspected a non-monotonic effect (for example the risk does not grow linearly with
the degree), the one-hot would leave the model more freedom.

</details>